# Análisis de relaciones entre dominios

Este notebook documenta la investigación sobre cómo se relacionan los tres dominios de datos (university, billing, crm): qué columnas los conectan realmente, qué se probó para buscar relaciones no declaradas, y qué se descartó con evidencia.

## Configuración

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("/home/jovyan/work")
RAW_PATH = PROJECT_ROOT / "data" / "raw"


def load_raw(domain, table):
    path = RAW_PATH / domain / f"{table}.csv"
    return pd.read_csv(path)


print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /home/jovyan/work


## 1. Relaciones dentro de cada dominio

Antes de buscar relaciones *entre* dominios, se listan las relaciones declaradas *dentro* de cada uno, inferidas de las columnas *_id compartidas entre tablas.

### University
| Tabla origen | Tabla destino | Columna |
|---|---|---|
| courses | professors | professor_id |
| enrollments | students | student_id |
| enrollments | courses | course_id |
| enrollments | semesters | semester_id |
| grades | enrollments | enrollment_id |

### Billing
| Tabla origen | Tabla destino | Columna |
|---|---|---|
| subscriptions | customers | customer_id |
| subscriptions | products | product_id |
| invoices | customers | customer_id |
| invoice_items | invoices | invoice_id |
| invoice_items | products | product_id |
| payments | invoices | invoice_id |

Finalizando el analisis se llega a la conclusion de que la relacion invoice_items --> products no es valida

### CRM
| Tabla origen | Tabla destino | Columna |
|---|---|---|
| contacts | accounts | account_id |
| opportunities | accounts | account_id |
| opportunity_contacts | opportunities | opportunity_id |
| opportunity_contacts | contacts | contact_id |
| activities | contacts | contact_id (opcional) |
| activities | opportunities | opportunity_id (opcional) |

leads no tiene ninguna columna *_id de salida: es una tabla aislada dentro de su propio dominio.

activities tiene dos FK opcionales e independientes entre sí: una fila puede tener contact_id, opportunity_id, ambas o ninguna. No es una relación excluyente.

## 2. Relación confirmada: University ↔ Billing

La única relación entre dominios mencionada en el README es billing.customers.external_ref ↔ university.students.student_id. Se verifica con datos.

In [2]:
students = load_raw("university", "students")
customers = load_raw("billing", "customers")

linked = customers[customers["external_ref"].notna()]
orphans = customers[customers["external_ref"].isna()]

print("Total customers:", len(customers))
print("Customers con external_ref:", len(linked))
print("Customers sin external_ref:", len(orphans))
print(
    "external_ref que existen en students.student_id:",
    linked["external_ref"].isin(students["student_id"]).sum(),
    "/",
    len(linked),
)
print("student_id cubiertos por algun customer:", linked["external_ref"].nunique(), "/", len(students))

Total customers: 10000
Customers con external_ref: 5000
Customers sin external_ref: 5000
external_ref que existen en students.student_id: 5000 / 5000
student_id cubiertos por algun customer: 5000 / 5000


De los 10,000 customers, exactamente 5,000 tienen external_ref y los 5,000 valores existen en students.student_id sin excepción, cubriendo el 100% de los students. Los otros 5,000 customers no tienen ningún vínculo con university. Es una relación real, pero parcial: cubre el 50% de los customers.

### 2.1 ¿Por qué external_ref y customer_id comparten el mismo número?

Los 5,000 customers vinculados tienen el mismo número que su external_ref (ej. CUS-0000001 → STU-0000001). Se verifica si esto es constante y, de ser así, si implica que el customer y el student son la misma persona.

El número idéntico no es una regla de negocio, es un artefacto de generación (customers y students se crearon en el mismo orden). El nombre completo coincide en apenas 1 de los 5,000 pares y el email no coincide en ninguno: el customer no es la misma persona que el student. Representa a quien paga (posible apoderado, tutor o empresa), mientras external_ref indica a nombre de qué estudiante es el pago.

## 3. Búsqueda de relación entre CRM y los demás dominios

Ni crm.accounts, crm.contacts, crm.leads ni crm.opportunities tienen una columna *_id que apunte a billing o university (ver sección 1). Se probaron varias formas de encontrar una relación implícita en el contenido de los datos.

### 3.1 Coincidencia exacta de email

Si una misma persona existiera en dos sistemas, lo más directo sería que comparta el mismo email.

In [3]:
contacts = load_raw("crm", "contacts")
leads = load_raw("crm", "leads")

stu_email = set(students["email"])
cus_email = set(customers["email"])
con_email = set(contacts["email"])
lead_email = set(leads["email"])

print("students --> customers:", len(stu_email & cus_email))
print("students --> contacts:", len(stu_email & con_email))
print("students --> leads:", len(stu_email & lead_email))
print("customers --> contacts:", len(cus_email & con_email))
print("customers --> leads:", len(cus_email & lead_email))
print("contacts --> leads:", len(con_email & lead_email))

students --> customers: 1
students --> contacts: 0
students --> leads: 0
customers --> contacts: 1
customers --> leads: 0
contacts --> leads: 0


Las coincidencias son 0 o 1 sobre miles de registros comparados en cada par, es decir, ruido y no una relación real. El email no conecta ningún dominio con crm.

### 3.2 Coincidencia de nombre completo (con grupo de control)

Se prueba si el nombre completo (first_name + last_name) sirve como llave. Se agrega un grupo de control: students contra contacts, dos tablas de las que ya sabemos que no tienen relación, para tener una referencia de cuánto "coincide por azar".

In [6]:
def nombre_set(df):
    return set(zip(df["first_name"], df["last_name"]))


stu_nom = nombre_set(students)
cus_nom = nombre_set(customers)
con_nom = nombre_set(contacts)

leads["nombre"] = list(zip(leads["first_name"], leads["last_name"]))
students["nombre"] = list(zip(students["first_name"], students["last_name"]))

print("Leads (2000) con nombre igual en contacts:", leads["nombre"].isin(con_nom).sum())
print("Leads (2000) con nombre igual en customers:", leads["nombre"].isin(cus_nom).sum())
print("Leads (2000) con nombre igual en students:", leads["nombre"].isin(stu_nom).sum())
print()
print("CONTROL - students (5000, sin relacion conocida) con nombre igual en contacts:", students["nombre"].isin(con_nom).sum())

Leads (2000) con nombre igual en contacts: 1997
Leads (2000) con nombre igual en customers: 1959
Leads (2000) con nombre igual en students: 1732

CONTROL - students (5000, sin relacion conocida) con nombre igual en contacts: 4989


El 99%+ de los registros de cualquier tabla tiene un homónimo en contacts, incluido el grupo de control (students vs contacts, sin relación conocida). El pool de nombres usado para generar los datos es pequeño (unos 2,000-2,500 nombres únicos) y se reutiliza en las 37,000 personas del dataset. El nombre no es una llave válida en ningún cruce, sea cual sea la tabla.

### 3.3 Partir el email y comparar solo la parte local

Se prueba si, ignorando el dominio del correo (@example.com, @demo.io, etc.), la parte local del email (antes del @) coincide entre dominios.

In [4]:
def local_part(email):
    return email.split("@")[0]


stu_local = set(students["email"].apply(local_part))
cus_local = set(customers["email"].apply(local_part))
con_local = set(contacts["email"].apply(local_part))
lead_local = set(leads["email"].apply(local_part))

print("students --> customers:", len(stu_local & cus_local))
print("students --> contacts:", len(stu_local & con_local))
print("customers --> contacts:", len(cus_local & con_local))
print("contacts --> leads:", len(con_local & lead_local))

students --> customers: 2
students --> contacts: 1
customers --> contacts: 5
contacts --> leads: 0


Las coincidencias siguen siendo mínimas (0 a 5 sobre miles de registros). Partir el email y comparar solo la parte local tampoco revela una relación oculta.

### 3.4 Timestamps exactos de creación

Se prueba si algún created_at (con hora y segundos) se repite exactamente entre tablas de distintos dominios, lo que indicaría que se generaron en el mismo instante.

In [5]:
accounts = load_raw("crm", "accounts")
opportunities = load_raw("crm", "opportunities")

cus_created = set(customers["created_at"])
con_created = set(contacts["created_at"])
acc_created = set(accounts["created_at"])
opp_created = set(opportunities["created_at"])

print("customers.created_at --> contacts.created_at:", len(cus_created & con_created))
print("customers.created_at --> accounts.created_at:", len(cus_created & acc_created))
print("customers.created_at --> opportunities.created_at:", len(cus_created & opp_created))

customers.created_at --> contacts.created_at: 1
customers.created_at --> accounts.created_at: 0
customers.created_at --> opportunities.created_at: 1


Como máximo 1 coincidencia exacta por par comparado, sobre decenas de miles de combinaciones posibles. Es ruido, no evidencia de relación.

In [ ]:
En base a todo lo demostrado, se concluye que CRM no comparte ninguna relacion con los demas dominios (university, billing)